# 01 · Errors & Exception Handling

**Module 07 – Errors, Debugging & Testing** · Estimated time: 70 min

> Merges 📗 *Errors and Exceptions Handling* with 📘 *Exception Handling and Custom Errors*, plus modern features: exception chaining, notes (3.11), exception groups & `except*` (3.11), and 3.14 syntax updates.

### 🎯 What you will learn
1. Syntax errors vs exceptions; reading a traceback
2. Common built-in exceptions and the exception hierarchy
3. `try` / `except` / `else` / `finally`
4. Catching multiple exceptions; accessing the exception object
5. Raising exceptions; re-raising; chaining with `raise ... from`
6. Custom exception classes and hierarchies
7. `assert`
8. EAFP vs LBYL
9. 🆕 `add_note()`, `ExceptionGroup`, `except*` (3.11)
10. 🆕 3.14: `except A, B:` without parentheses; warnings for `return` in `finally`

## 1. Two kinds of errors

**Syntax errors**: the code isn't valid Python, so nothing runs:

In [1]:
print('Hello)

SyntaxError: unterminated string literal (detected at line 1) (1679058590.py, line 1)

**Exceptions**: the code is valid but something goes wrong **while running**:

In [2]:
numbers = [1, 2, 3]
print(numbers[10])

IndexError: list index out of range

### Reading a traceback
Read it **bottom-up**:
1. **Last line**: exception type + message (`IndexError: list index out of range`)
2. Lines above: *where* it happened (file, line number, code), most recent call last

In [3]:
def level3(x):
    return 10 / x

def level2(x):
    return level3(x) + 1

def level1():
    return level2(0)

level1()

ZeroDivisionError: division by zero

> 🆕 Python 3.11+ adds `^^^^^` markers pointing at the exact failing expression, and 3.13+ colours tracebacks in the terminal.

## 2. Common built-in exceptions

In [4]:
examples = {
    "ZeroDivisionError": "1 / 0",
    "TypeError": "'2' + 2",
    "ValueError": "int('abc')",
    "NameError": "undefined_variable",
    "IndexError": "[1, 2][5]",
    "KeyError": "{'a': 1}['b']",
    "AttributeError": "'text'.push('x')",
    "FileNotFoundError": "open('no_such_file.txt')",
    "ModuleNotFoundError": "__import__('no_such_module')",
    "StopIteration": "next(iter([]))",
}
for name, code in examples.items():
    try:
        eval(code)
    except Exception as e:
        print(f"{type(e).__name__:<22} ← {code:<32} {e}")

ZeroDivisionError      ← 1 / 0                            division by zero
TypeError              ← '2' + 2                          can only concatenate str (not "int") to str
ValueError             ← int('abc')                       invalid literal for int() with base 10: 'abc'
NameError              ← undefined_variable               name 'undefined_variable' is not defined
IndexError             ← [1, 2][5]                        list index out of range
KeyError               ← {'a': 1}['b']                    'b'
AttributeError         ← 'text'.push('x')                 'str' object has no attribute 'push'
FileNotFoundError      ← open('no_such_file.txt')         [Errno 2] No such file or directory: 'no_such_file.txt'
ModuleNotFoundError    ← __import__('no_such_module')     No module named 'no_such_module'
StopIteration          ← next(iter([]))                   


### The exception hierarchy (simplified)

```
BaseException
 ├── SystemExit, KeyboardInterrupt, GeneratorExit   ← don't catch these normally
 └── Exception                                      ← catch from here down
      ├── ArithmeticError → ZeroDivisionError, OverflowError
      ├── LookupError → IndexError, KeyError
      ├── OSError → FileNotFoundError, PermissionError, TimeoutError, ...
      ├── ValueError → UnicodeError
      ├── TypeError, NameError, AttributeError, ImportError, RuntimeError, ...
      └── ExceptionGroup (3.11)
```
Catching a parent catches all its children:

In [5]:
try:
    {"a": 1}["z"]
except LookupError as e:          # catches KeyError AND IndexError
    print("Lookup failed:", repr(e))

Lookup failed: KeyError('z')


## 3. `try` / `except`

```python
try:
    # code that might fail
except SomeError:
    # runs only if SomeError happened in the try block
```

In [6]:
try:
    x = 10 / 0
except ZeroDivisionError:
    print("Cannot divide by zero!")
print("The program keeps running")

Cannot divide by zero!
The program keeps running


### Accessing the exception with `as`

In [7]:
try:
    int("forty-two")
except ValueError as e:
    print("Type:", type(e).__name__)
    print("Message:", e)
    print("Args:", e.args)

Type: ValueError
Message: invalid literal for int() with base 10: 'forty-two'
Args: ("invalid literal for int() with base 10: 'forty-two'",)


### Multiple exceptions (📘)

In [8]:
def safe_divide(a, b):
    try:
        return float(a) / float(b)
    except ZeroDivisionError:
        return "You can't divide by zero!"
    except ValueError:
        return "Invalid input! Please enter numbers."

print(safe_divide(10, 2), safe_divide(10, 0), safe_divide("ten", 2), sep=" | ")

5.0 | You can't divide by zero! | Invalid input! Please enter numbers.


In [9]:
# Several exception types handled the same way: a tuple
try:
    result = 10 / int("0")
except (ZeroDivisionError, ValueError) as e:
    print(f"An error occurred: {e}")

An error occurred: division by zero


🆕 **Python 3.14 (PEP 758):** the parentheses are optional **when you don't use `as`**:

In [10]:
try:
    int("x")
except ValueError, TypeError:
    print("caught (3.14+ syntax)")

caught (3.14+ syntax)


### ❌ Avoid bare `except:`
A bare `except:` (or `except BaseException:`) catches *everything*, including `KeyboardInterrupt` (Ctrl+C) and typos in your own code, hiding bugs. Catch **specific** exceptions; if you must catch broadly, use `except Exception as e:` and log it.

## 4. `else` and `finally`

- `else`: runs only if **no exception** occurred in `try`
- `finally`: **always** runs (success, exception, even `return`); perfect for cleanup

In [11]:
def read_number(text):
    try:
        value = int(text)
    except ValueError:
        print(f"  {text!r} is not a number")
        return None
    else:
        print(f"  Parsed {value} successfully")
        return value
    finally:
        print("  (finally: always runs)")

print(read_number("42"))
print(read_number("abc"))

  Parsed 42 successfully
  (finally: always runs)
42
  'abc' is not a number
  (finally: always runs)
None


### 📗 Example: writing to files

In [12]:
try:
    f = open("testfile", "w")
    f.write("Test write this")
except OSError:                       # ⚠️ the bootcamp uses IOError, now just an alias of OSError
    print("Error: Could not find file or read data")
else:
    print("Content written successfully")
    f.close()

Content written successfully


In [13]:
try:
    f = open("testfile", "r")         # opened read-only
    f.write("Test write this")        # → io.UnsupportedOperation (a subclass of OSError)
except OSError:
    print("Error: Could not find file or read data")
else:
    print("Content written successfully")
    f.close()
finally:
    print("Always execute finally code blocks")

Error: Could not find file or read data
Always execute finally code blocks


> ⚠️ **Old pattern (📘 handbook):**
> ```python
> try:
>     file = open("test.txt", "r")
>     content = file.read()
> except FileNotFoundError: ...
> finally:
>     file.close()   # ❌ NameError if open() itself failed!
> ```
> ✅ Use `with open(...) as file:`: it closes the file automatically, even on errors.

### 📗 Keep asking until the input is valid

In [14]:
def askint(answers):
    for answer in answers:                 # stands in for repeated input() calls
        try:
            val = int(answer)
        except ValueError:
            print(f"{answer!r}: Looks like you did not enter an integer!")
            continue
        else:
            print("Yep that's an integer!", val)
            return val
        finally:
            print("Finally, I executed!")

askint(["null", "3.5", "7"])

'null': Looks like you did not enter an integer!
Finally, I executed!
'3.5': Looks like you did not enter an integer!
Finally, I executed!
Yep that's an integer! 7
Finally, I executed!


7

Real version with `input()`:
```python
def askint():
    while True:
        try:
            val = int(input("Please enter an integer: "))
        except ValueError:
            print("Looks like you did not enter an integer!")
            continue
        else:
            print("Yep that's an integer!")
            return val
```

🆕 **3.14 (PEP 765):** `return`, `break` or `continue` *inside* a `finally` block now emits a `SyntaxWarning`, because it silently swallows exceptions. Don't do it.

## 5. Raising exceptions

In [15]:
def check_age(age):
    if not isinstance(age, int):
        raise TypeError(f"age must be an int, got {type(age).__name__}")
    if age < 18:
        raise ValueError("Age must be 18 or older!")
    return "Access granted."

for a in [20, 16, "twenty"]:
    try:
        print(check_age(a))
    except (ValueError, TypeError) as e:
        print(f"Error: {e}")

Access granted.
Error: Age must be 18 or older!
Error: age must be an int, got str


### Re-raising: handle partially, then pass it on

In [16]:
def load_config():
    try:
        return {"a": 1}["missing"]
    except KeyError:
        print("log: config key missing")
        raise                       # re-raise the SAME exception

try:
    load_config()
except KeyError as e:
    print("caller saw:", repr(e))

log: config key missing
caller saw: KeyError('missing')


### Exception chaining with `raise ... from`

Translate a low-level error into a meaningful one, **keeping the original cause**:

In [17]:
class ConfigError(Exception):
    pass

def parse_port(text):
    try:
        return int(text)
    except ValueError as e:
        raise ConfigError(f"Invalid port {text!r}") from e

parse_port("eighty")

ConfigError: Invalid port 'eighty'

## 6. Custom exceptions (📘)

Create your own by subclassing `Exception`. This makes errors specific, informative and easy to catch selectively.

In [18]:
class InvalidAgeError(Exception):
    """Raised when an age is below the allowed minimum."""
    def __init__(self, age, minimum=18):
        self.age = age
        self.minimum = minimum
        super().__init__(f"Age {age} is below the minimum of {minimum}")

def verify_age(age):
    if age < 18:
        raise InvalidAgeError(age)
    return "Welcome!"

try:
    print(verify_age(16))
except InvalidAgeError as e:
    print(f"Error: {e} (age={e.age})")

Error: Age 16 is below the minimum of 18 (age=16)


### An exception hierarchy for an application

In [19]:
class BankError(Exception):
    """Base class for all banking errors."""

class InsufficientFundsError(BankError):
    def __init__(self, balance, amount):
        super().__init__(f"Cannot withdraw {amount}; balance is only {balance}")
        self.shortfall = amount - balance

class AccountLockedError(BankError):
    pass

def withdraw(balance, amount, locked=False):
    if locked:
        raise AccountLockedError("Account is locked")
    if amount > balance:
        raise InsufficientFundsError(balance, amount)
    return balance - amount

for args in [(100, 50), (100, 500), (100, 10, True)]:
    try:
        print("New balance:", withdraw(*args))
    except InsufficientFundsError as e:
        print(f"{e} (short by {e.shortfall})")
    except BankError as e:                 # catches any other bank error
        print("Bank error:", e)

New balance: 50
Cannot withdraw 500; balance is only 100 (short by 400)
Bank error: Account is locked


## 7. `assert`: sanity checks for developers

In [20]:
def average(nums):
    assert len(nums) > 0, "nums must not be empty"
    return sum(nums) / len(nums)

print(average([1, 2, 3]))
average([])

2.0


AssertionError: nums must not be empty

> ⚠️ Assertions are removed when Python runs with `-O`. **Never** use `assert` to validate user input or security checks; raise a real exception instead.

## 8. EAFP vs LBYL

- **LBYL** (*Look Before You Leap*): check first → `if key in d: ...`
- **EAFP** (*Easier to Ask Forgiveness than Permission*): try it, handle failure → Pythonic, and avoids race conditions

In [21]:
d = {"name": "Asha"}
# LBYL
if "age" in d:
    print(d["age"])
else:
    print("no age")
# EAFP
try:
    print(d["age"])
except KeyError:
    print("no age")

no age
no age


## 9. 🆕 Exception notes (3.11)

Add context to an exception as it propagates:

In [22]:
try:
    try:
        int("x1")
    except ValueError as e:
        e.add_note("while parsing row 17 of data.csv")
        raise
except ValueError as e:
    print(e)
    print(e.__notes__)

invalid literal for int() with base 10: 'x1'
['while parsing row 17 of data.csv']


## 10. 🆕 Exception groups and `except*` (3.11)

Sometimes **several** errors happen at once (e.g. validating many fields, or concurrent tasks in `asyncio.TaskGroup`). An `ExceptionGroup` bundles them; `except*` handles each type separately.

In [23]:
def validate(user):
    errors = []
    if not user.get("name"):
        errors.append(ValueError("name is required"))
    if not isinstance(user.get("age"), int):
        errors.append(TypeError("age must be an int"))
    if "@" not in user.get("email", ""):
        errors.append(ValueError("email is invalid"))
    if errors:
        raise ExceptionGroup("invalid user", errors)

try:
    validate({"name": "", "age": "x", "email": "nope"})
except* ValueError as eg:
    for e in eg.exceptions:
        print("ValueError:", e)
except* TypeError as eg:
    for e in eg.exceptions:
        print("TypeError:", e)

ValueError: name is required
ValueError: email is invalid
TypeError: age must be an int


## 11. Best practices
- ✅ Catch **specific** exceptions; keep `try` blocks **small**
- ✅ Use `with` for resources instead of manual `finally: close()`
- ✅ Create a base exception for your app/library; subclass it
- ✅ Use `raise ... from e` when translating exceptions
- ✅ Log unexpected exceptions (`logging.exception(...)`, Module 07 lesson 02)
- ❌ Don't silence errors with `except: pass`
- ❌ Don't use exceptions for normal control flow where an `if` is clearer

## 📝 Summary
- `try/except/else/finally`; `except (A, B) as e`; 3.14 allows `except A, B:` without `as`.
- `raise` to signal, bare `raise` to re-raise, `raise X from e` to chain.
- Custom exceptions subclass `Exception`; build hierarchies.
- 🆕 `add_note()`, `ExceptionGroup` + `except*`.

## 🏋️ Practice (📗 Errors homework, full version in `exercises/`)
1. Handle the exception thrown by `for i in ['a', 'b', 'c']: print(i ** 2)`, printing an error message.
2. Handle the exception for `x = 5; y = 0; z = x / y` and print "All Done." in a `finally` block.
3. Write `ask()` that keeps asking for an integer and prints its square (use a loop + `try/except/else`).
4. Create `NegativeNumberError` and raise it from `sqrt(n)` for negative inputs.

<details><summary>Show solutions</summary>

```python
try:
    for i in ['a', 'b', 'c']:
        print(i ** 2)
except TypeError:
    print("There was a type error!")

try:
    x = 5; y = 0
    z = x / y
except ZeroDivisionError:
    print("Can't divide by zero!")
finally:
    print("All Done.")

def ask():
    while True:
        try:
            n = int(input("Input an integer: "))
        except ValueError:
            print("An error occurred! Please try again!")
        else:
            print("Thank you, your number squared is:", n ** 2)
            break

class NegativeNumberError(ValueError):
    pass
def sqrt(n):
    if n < 0:
        raise NegativeNumberError(f"{n} is negative")
    return n ** 0.5
```
</details>